In [1]:
import pandas as pd ,numpy as np , matplotlib.pyplot as plt, seaborn as sns

In [2]:
from pathlib import Path

PROJECT = Path(".")          # current folder = project folder
out = PROJECT / "outputs"
out.mkdir(exist_ok=True)

df = pd.read_csv(PROJECT / "emails.csv")
df.head()

,text,spam
0,Subject: naturally irresistible your corporate...,1
1,Subject: the stock trading gunslinger fanny i...,1
2,Subject: unbelievable new homes made easy im ...,1
3,Subject: 4 color printing special request add...,1
4,"Subject: do not have money , get software cds ...",1


In [3]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5728 entries, 0 to 5727
Data columns (total 2 columns):
 #   Column  Non-Null Count  Dtype 
---  ------  --------------  ----- 
 0   text    5728 non-null   object
 1   spam    5728 non-null   int64 
dtypes: int64(1), object(1)
memory usage: 89.6+ KB


In [4]:
df.isnull().sum()

text    0
spam    0
dtype: int64

In [5]:
df.spam.value_counts()

spam
0    4360
1    1368
Name: count, dtype: int64

In [6]:
not_spam = (df["spam"] == 0).sum()
spam = (df["spam"] == 1).sum()
df["words"] = df["text"].astype(str).str.split().str.len()

In [7]:
plt.figure(figsize=(5, 4))
plt.bar(["Not spam (0)", "Spam (1)"], [not_spam, spam], color=["#143330", "#18A18F"])
plt.title("Spam vs Not spam")
plt.ylabel("Count")
plt.savefig(out / "1_spam_vs_not_spam.png", dpi=150)
plt.close()

In [8]:
plt.figure(figsize=(5, 4))
plt.pie(
    [not_spam, spam],
    labels=["Not spam (0)", "Spam (1)"],
    autopct="%1.1f%%",
    colors=["#143330", "#18A18F"],
)
plt.title("0 = Not spam, 1 = Spam")
plt.savefig(out / "2_pie.png", dpi=150)
plt.close()

In [9]:
plt.figure(figsize=(6, 4))
plt.hist(df.loc[df["spam"] == 0, "words"], bins=30, alpha=0.7, label="Not spam (0)", color="#143330")
plt.hist(df.loc[df["spam"] == 1, "words"], bins=30, alpha=0.7, label="Spam (1)", color="#18A18F")
plt.title("Words per email")
plt.xlabel("Word count")
plt.legend()
plt.savefig(out / "3_word_length.png", dpi=150)
plt.close()

In [10]:
# Step 2: Text preprocessing (lowercase, remove punctuation, stopwords, stemming)
import re
import nltk
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer
from nltk.tokenize import word_tokenize

nltk.download("stopwords", quiet=True)
nltk.download("punkt", quiet=True)
nltk.download("punkt_tab", quiet=True)

stop_words = set(stopwords.words("english"))
stemmer = PorterStemmer()



In [11]:
def preprocess_text(text):
    """0 = not spam, 1 = spam — clean raw email text for the model."""
    text = str(text).lower()
    text = re.sub(r"[^a-z\s]", " ", text)
    words = word_tokenize(text)
    words = [stemmer.stem(w) for w in words if w not in stop_words and len(w) > 1]
    return " ".join(words)


df["clean_text"] = df["text"].apply(preprocess_text)
print("Preprocessing done. Sample:")
print("BEFORE:", df["text"].iloc[0][:200], "...")
print("AFTER: ", df["clean_text"].iloc[0][:200], "...")

Preprocessing done. Sample:
BEFORE: Subject: naturally irresistible your corporate identity  lt is really hard to recollect a company : the  market is full of suqgestions and the information isoverwhelminq ; but a good  catchy logo , st ...
AFTER:  subject natur irresist corpor ident lt realli hard recollect compani market full suqgest inform isoverwhelminq good catchi logo stylish statloneri outstand websit make task much easier promis havinq o ...


In [12]:
# Step 4: Train/test split (on clean_text, not raw text) — stratify keeps spam % same
from sklearn.model_selection import train_test_split

X = df["clean_text"]
y = df["spam"]  # 0 = not spam, 1 = spam

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print("Train size:", len(X_train))
print("Test size:", len(X_test))
print("Spam in train:", y_train.sum(), "| Spam in test:", y_test.sum())

Train size: 4582
Test size: 1146
Spam in train: 1094 | Spam in test: 274


In [13]:
# Step 3: TF-IDF features — fit on TRAIN only, then transform test
# ngram_range=(1,2) uses single words + pairs (e.g. "click here") for better spam detection
from sklearn.feature_extraction.text import TfidfVectorizer

vectorizer = TfidfVectorizer(
    max_features=8000,
    ngram_range=(1, 2),
    min_df=2,
    sublinear_tf=True,
)
X_train_vec = vectorizer.fit_transform(X_train)
X_test_vec = vectorizer.transform(X_test)

print("Train matrix:", X_train_vec.shape)
print("Test matrix:", X_test_vec.shape)

Train matrix: (4582, 8000)
Test matrix: (1146, 8000)


In [14]:
# Step 5: Train Multinomial Naive Bayes
# class_prior balances spam vs ham (dataset has more ham emails)
from sklearn.naive_bayes import MultinomialNB

model = MultinomialNB(alpha=0.1, class_prior=[0.5, 0.5], fit_prior=False)
model.fit(X_train_vec, y_train)
y_pred = model.predict(X_test_vec)

print("Model trained.")

Model trained.


In [15]:
# Step 6: Evaluation — accuracy, precision, recall, confusion matrix
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
    confusion_matrix,
)

print("Accuracy: ", accuracy_score(y_test, y_pred))
print("Precision:", precision_score(y_test, y_pred))
print("Recall:   ", recall_score(y_test, y_pred))
print("F1 score: ", f1_score(y_test, y_pred))
print()




Accuracy:  0.987783595113438
Precision: 0.9642857142857143
Recall:    0.9854014598540146
F1 score:  0.9747292418772563



In [16]:
print(classification_report(y_test, y_pred, target_names=["Not spam (0)", "Spam (1)"]))

              precision    recall  f1-score   support

Not spam (0)       1.00      0.99      0.99       872
    Spam (1)       0.96      0.99      0.97       274

    accuracy                           0.99      1146
   macro avg       0.98      0.99      0.98      1146
weighted avg       0.99      0.99      0.99      1146



In [17]:
cm = confusion_matrix(y_test, y_pred)
plt.figure(figsize=(5, 4))
sns.heatmap(
    cm,
    annot=True,
    fmt="d",
    cmap="Blues",
    xticklabels=["Not spam", "Spam"],
    yticklabels=["Not spam", "Spam"],
)
plt.title("Confusion Matrix")
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.tight_layout()
plt.savefig(out / "4_confusion_matrix.png", dpi=150)
plt.close()
print("Saved:", out / "4_confusion_matrix.png")


Saved: outputs\4_confusion_matrix.png


In [18]:
# Step 7: Predict on your own text
def predict_spam(message):
    """Print prediction only (no return value — avoids extra Jupyter output)."""
    clean = preprocess_text(message)
    vec = vectorizer.transform([clean])
    proba = model.predict_proba(vec)[0]  # [P(not spam), P(spam)]
    pred = 1 if proba[1] >= 0.5 else 0
    label = "Spam (1)" if pred == 1 else "Not spam (0)"
    print(f"{label}  (spam confidence: {proba[1]:.2f})")

In [19]:
# Test custom messages (loop avoids Jupyter showing a 4th "return" line)
test_messages = [
    "Congratulations! You won $1000. Click here now.",
    "Meeting is on Tuesday at 3pm. Thanks.",
    "Free viagra now!!! Limited offer click link",
]

for message in test_messages:
    predict_spam(message)

Spam (1)  (spam confidence: 0.60)
Not spam (0)  (spam confidence: 0.01)
Spam (1)  (spam confidence: 1.00)
